# ML GPR Dataset Workflow

This notebook gives a full pre-model and model workflow for the `bcc`, `fcc`, and `hcp` NPZ datasets.

- The pre-model part does not import `graphdot`.
- The training part is optional and only runs if you set `RUN_TRAINING = True`.
- You can use either `PHASES` or explicit `INPUTS`.


In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
from IPython.display import Image, display


# Set these only if Jupyter starts outside the project tree.
MANUAL_PROJECT_HINT = None
# Example: MANUAL_PROJECT_HINT = Path("/Users/dajuarez4/Documents/Fe")
MANUAL_DATASET_HINT = None


def find_project_paths(start=None, project_hint=None, dataset_hint=None):
    candidate_starts = []
    if project_hint is not None:
        candidate_starts.append(Path(project_hint).expanduser().resolve())
    if start is not None:
        candidate_starts.append(Path(start).expanduser().resolve())
    candidate_starts.extend([
        Path.cwd().resolve(),
        Path("/Users/dajuarez4/Documents/Fe"),
        Path("/Users/dajuarez4/Documents/Fe/IronCoreMD"),
    ])

    seen = set()
    for start_dir in candidate_starts:
        for candidate in (start_dir, *start_dir.parents):
            candidate = candidate.resolve()
            if candidate in seen:
                continue
            seen.add(candidate)

            repo_root = None
            work_root = None
            if (candidate / "IronCoreMD" / "codes" / "ml_gpr.py").exists():
                work_root = candidate
                repo_root = candidate / "IronCoreMD"
            elif (candidate / "codes" / "ml_gpr.py").exists():
                repo_root = candidate
                work_root = candidate.parent

            if repo_root is None:
                continue

            dataset_candidates = []
            if dataset_hint is not None:
                dataset_candidates.append(Path(dataset_hint).expanduser().resolve())
            dataset_candidates.extend([
                work_root / "dataset",
                repo_root / "dataset",
            ])

            for dataset_root in dataset_candidates:
                if dataset_root.exists():
                    return work_root, repo_root, dataset_root, repo_root / "codes"

    raise FileNotFoundError(
        "Could not locate the project paths. Set MANUAL_PROJECT_HINT or MANUAL_DATASET_HINT explicitly."
    )


def load_module(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


WORK_ROOT, REPO_ROOT, DATASET_ROOT, CODES_DIR = find_project_paths(
    project_hint=MANUAL_PROJECT_HINT,
    dataset_hint=MANUAL_DATASET_HINT,
)
PREVIEW_PATH = CODES_DIR / "plot_ml_dataset_split.py"
ML_GPR_PATH = CODES_DIR / "ml_gpr.py"

pre_model = load_module("plot_ml_dataset_split_notebook", PREVIEW_PATH)
pre_model.PROJECT_ROOT = WORK_ROOT
pre_model.DEFAULT_DATASET_PATTERNS = {
    "bcc": str(DATASET_ROOT / "bcc" / "non-mag" / "*.npz"),
    "fcc": str(DATASET_ROOT / "fcc" / "non-mag" / "*.npz"),
    "hcp": str(DATASET_ROOT / "hcp" / "*.npz"),
}

print("Workspace root:", WORK_ROOT)
print("Repo root:", REPO_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Codes dir:", CODES_DIR)
print("Preview module:", PREVIEW_PATH)
print("Training module:", ML_GPR_PATH)


## Configuration

Use either:

- `PHASES = ["bcc", "fcc", "hcp"]` to select full phase groups, or
- `INPUTS = [...]` to pass explicit NPZ files or glob patterns.

If `INPUTS` is not `None`, it overrides `PHASES`.


In [ ]:
PHASES = ["bcc", "fcc", "hcp"]
INPUTS = None

# Example explicit inputs:
# INPUTS = [
#     str(DATASET_ROOT / "bcc" / "non-mag" / "2.29_5000K.npz"),
#     str(DATASET_ROOT / "fcc" / "non-mag" / "3.00_5000K.npz"),
#     str(DATASET_ROOT / "hcp" / "a_2.16_c_3.42_5000K.npz"),
# ]

RUN_NAME = "bcc_fcc_hcp_demo"
OUTPUT_ROOT = REPO_ROOT / "ml-results"
SEED = 0
RUN_TRAINING = False
TRAIN_PREVIEW_ONLY = False


In [ ]:
preview_args = pre_model.build_runtime_args(
    phase=PHASES,
    inputs=INPUTS,
    run_name=RUN_NAME,
    output_root=str(OUTPUT_ROOT),
    seed=SEED,
)

input_files = pre_model.resolve_input_files(preview_args)
print("Resolved files:")
for file_path in input_files:
    print("  ", file_path)
print("Total files:", len(input_files))


In [ ]:
dataset_df = pre_model.run_pre_model_split(
    phase=PHASES,
    inputs=INPUTS,
    run_name=RUN_NAME,
    output_root=str(OUTPUT_ROOT),
    seed=SEED,
)


In [ ]:
display(dataset_df.head())
print("\nSplit counts:")
print(dataset_df["split"].value_counts())

print("\nCounts by phase and split:")
display(dataset_df.groupby(["phase", "split"]).size().unstack(fill_value=0))


In [ ]:
base_name = pre_model.build_run_name(preview_args, input_files)
out_dir = pre_model.build_output_dir(preview_args, base_name)

dataset_csv = out_dir / f"{base_name}_energy_volume_dataset.csv"
train_csv = out_dir / f"{base_name}_energy_volume_train.csv"
test_csv = out_dir / f"{base_name}_energy_volume_test.csv"
plot_png = out_dir / f"{base_name}_energy_vs_volume_train_test.png"

print("Output directory:", out_dir)
print("Dataset CSV:", dataset_csv)
print("Train CSV:", train_csv)
print("Test CSV:", test_csv)
print("Preview plot:", plot_png)

display(Image(filename=str(plot_png)))


## Optional GPR Training

This part imports `ml_gpr.py`, which requires `graphdot` and its dependencies.
For the actual fit, the runtime environment also needs CUDA with `nvcc` available in `PATH`.

Leave `RUN_TRAINING = False` if you only want the pre-model dataset split.

If `TRAIN_PREVIEW_ONLY = True`, the ML script will build the split files and stop before fitting. In that case `result` is expected to be `None`.


In [ ]:
ml_gpr = None
try:
    ml_gpr = load_module("ml_gpr_notebook", ML_GPR_PATH)
    ml_gpr.PROJECT_ROOT = WORK_ROOT
    ml_gpr.DEFAULT_DATASET_PATTERNS = {
        "bcc": str(DATASET_ROOT / "bcc" / "non-mag" / "*.npz"),
        "fcc": str(DATASET_ROOT / "fcc" / "non-mag" / "*.npz"),
        "hcp": str(DATASET_ROOT / "hcp" / "*.npz"),
    }
    print("Loaded ml_gpr.py successfully.")
except Exception as exc:
    print("Could not load ml_gpr.py:")
    print(exc)


In [ ]:
result = None

if RUN_TRAINING:
    if ml_gpr is None:
        raise RuntimeError("ml_gpr.py could not be imported. Check graphdot and related dependencies.")

    result = ml_gpr.run_ml_gpr(
        phase=PHASES,
        inputs=INPUTS,
        run_name=RUN_NAME,
        output_root=str(OUTPUT_ROOT),
        seed=SEED,
        preview_only=TRAIN_PREVIEW_ONLY,
    )

    if TRAIN_PREVIEW_ONLY:
        print("TRAIN_PREVIEW_ONLY is True. Dataset preview files were created and no fitted model is returned.")
else:
    print("RUN_TRAINING is False. Set it to True to fit the model.")


In [ ]:
if result is not None:
    _, _, _, _, model_out_dir, model_base_name = result
    parity_png = Path(model_out_dir) / f"{model_base_name}_parity_plot.png"
    print("Parity plot:", parity_png)
    display(Image(filename=str(parity_png)))
elif RUN_TRAINING and TRAIN_PREVIEW_ONLY:
    print("No parity plot is expected because TRAIN_PREVIEW_ONLY is True.")
elif RUN_TRAINING:
    print("Training was requested, but no trained model result was returned.")
else:
    print("No trained model result available.")
